# 21. Bias, variance, overfitting, and model selection

**Level:** Intermediate

**Prerequisites:** 11-20: splits, models, and metrics.

[Course index](00_START_HERE.ipynb) | [Previous](20_Metrics_Thresholds_and_Calibration.ipynb) | [Next](22_KMeans_and_DBSCAN.ipynb)

**Topics:** Bias, Variance, Overfitting, Underfitting, Regularization, Cross-validation, Hyperparameters.

**Matching script:** `examples\21_Bias_Variance_Overfitting_and_Tuning.py`

**How to study:** read the notes, predict an example's output, run it, trace each change in data/shape/state, then complete the practice task. Every worked code cell contains its own imports and data and can run independently of other notebooks.

**Execution contract:** Python 3.11+, course dependencies, CPU only, synthetic/local data. No hosted API requests, model-weight downloads, dataset downloads, listening web servers, credentials, or real device changes. The temporary-file lesson cleans up its own files. Expected errors are caught narrowly and asserted; unexpected errors must remain visible.

## Concept notes and mental model

Underfitting means a model fails to capture useful structure, often producing high training
and validation error. Overfitting means training fit captures noise or quirks that do not
generalize, often with a gap between training and validation performance.

Bias is systematic error from restrictive assumptions; variance is sensitivity to the sampled
training data. These are related to, but not identical with, training and validation scores.
Noise and distribution shift also affect performance. The familiar squared-error decomposition
assumes a particular statistical setting, not every classification metric.

Regularization, simpler models, data quality, and more representative data can help. A learning
curve varies training size; a validation curve varies a hyperparameter. Cross-validation
repeats training/validation splits on development data; tune preprocessing and estimation
together. Grouped or temporal folds are needed when random folds violate independence.

GridSearchCV selects a setting using inner validation scores. Its best_score_ is not an
untouched final test result. Evaluate the selected model on a held-out test only once.

## Mathematics or data contract

For squared error under standard assumptions, expected error includes bias^2 + variance + irreducible noise.

## Important limitation

Random cross-validation here assumes independent synthetic samples.

Distinguish what is implemented and asserted below from what would need additional data, training, independent evaluation, authorization, or operational controls.

## Worked example 1: Tune a polynomial ridge model without leaking the test set

**Purpose:** Separate selection from final evaluation.

### Read the code in this order
1. Generate noisy nonlinear data and reserve a test set.
2. Put transforms in a pipeline.
3. Search a tiny grid on training folds.
4. Evaluate the selected estimator on the untouched test partition.

### Syntax and behavior to notice
- `fit(...)` learns model or preprocessing state from the supplied training data.
- `predict(...)` returns the estimator's chosen output, not a guarantee of correctness.
- `assert` checks the stated behavior; a failed check raises AssertionError.
- A fixed random seed makes the chosen sampling reproducible within the environment.

### Expected output
This is the output captured from executing the worked example in the recorded environment. Package versions and floating-point rounding can change numeric details; the assertions express the required shapes, invariants, and thresholds.

```text
selected: {'poly__degree': 2, 'ridge__alpha': 0.01}
test MAE: 0.113
```

**Before running:** identify the input variables, which values are learned versus chosen, and what each assertion proves. A successful shape test does not by itself prove predictive quality. Training-only loss checks are labeled as such.

In [1]:
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

rng = np.random.default_rng(42)
X = rng.uniform(-2, 2, (120, 1))
y = X[:, 0] ** 2 + rng.normal(0, 0.15, 120)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
pipeline = Pipeline([("poly", PolynomialFeatures(include_bias=False)),
                     ("scale", StandardScaler()), ("ridge", Ridge())])
search = GridSearchCV(pipeline, {"poly__degree": [1, 2, 4], "ridge__alpha": [0.01, 1.]},
                      cv=KFold(3, shuffle=True, random_state=42),
                      scoring="neg_mean_absolute_error", n_jobs=1)
search.fit(X_train, y_train)
error = mean_absolute_error(y_test, search.predict(X_test))
assert error < 0.5
print("selected:", search.best_params_)
print("test MAE:", round(error, 3))

selected: {'poly__degree': 2, 'ridge__alpha': 0.01}
test MAE: 0.113


### Interpret example 1

The result should demonstrate this purpose: **Separate selection from final evaluation.**

Follow the assertions back to the exact data or state being checked. Where a model was fitted, distinguish training fit from held-out evaluation. Where input is rejected, inspect whether rejection happens before any intended mutation or tool execution. Do not delete a failed assertion to make an incorrect result look valid.

## Practice: extend the concept yourself

Compare polynomial degrees and plot train/validation errors; leave final test data out of the selection loop.

1. Write your expected result before editing code.
2. Add one normal-case test and one boundary/failure-case test.
3. Keep training/validation/test responsibilities separate where relevant.
4. Report an honest failure or unknown result rather than a success-shaped fallback.

The task is intentionally not filled in for you; the worked examples above are complete and runnable. Add your own code cell for the extension.

## Understanding checkpoint
- Explain `Bias` in your own words and give one example of when it is useful.
- Trace one worked example from input through transformation to returned or printed output.
- State this lesson's mathematical/data contract: For squared error under standard assumptions, expected error includes bias^2 + variance + irreducible noise.
- Name the boundary of the demonstration: Random cross-validation here assumes independent synthetic samples.
- Describe one test that could catch a plausible but wrong implementation.

## Primary references and further reading

- `https://scikit-learn.org/stable/user_guide.html`

Research snapshot: **2026-10-17**. Foundational references need not be recent. Mutable library docs describe their current version; compare with the tested dependency versions in the course manifest. References are reading material, not runtime downloads.